#  RMIFO - Demonstração Interativa
## Resilience of Intercontinental Fiber Optic Networks

**Análise de Resiliência de Redes de Fibra Ótica Intercontinentais com PySpark + GraphFrames**

###  O que vamos fazer:
-  Carregar dados de topologia de rede
-  Construir grafo de infraestrutura
-  Analisar criticidade de nós/cabos
-  Simular falhas e quantificar impactos
-  Visualizar resultados

> ** Dica**: Execute cada célula sequencialmente para ver o código em ação!

##  Passo 1: Setup & Instalação de Dependências

Se estiver no **Google Colab**, execute a célula abaixo para instalar PySpark e outras dependências:

In [ ]:
import sys
import subprocess

# Verifica se está no Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print(" Google Colab detectado! Instalando dependências...")
    
    # Instala PySpark
    subprocess.run(['pip', 'install', '-q', 'pyspark>=3.2.0'], check=True)
    
    # Instala GraphFrames
    subprocess.run(['pip', 'install', '-q', 'graphframes'], check=True)
    
    # Instala outras dependências
    subprocess.run(['pip', 'install', '-q', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'pyyaml'], check=True)
    
    print(" Dependências instaladas com sucesso!")
else:
    print(" Executando em ambiente local (não Colab)")

##  Passo 2: Importar Bibliotecas e Modelos RMIFO

In [ ]:
# Imports padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# Spark e GraphFrames
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, count, rand

print(" Todas as bibliotecas importadas com sucesso!")

# Configurar matplotlib para exibição melhor
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*60)
print("RMIFO - Framework Modularizado para Análise de Rede")
print("="*60)

##  Passo 3: Configuração e Inicialização do Framework

Aqui definimos as classes principais do framework RMIFO de forma modularizada:

In [ ]:
from dataclasses import asdict

# ============================================================================
# CLASSE 1: CONFIGURAÇÃO CENTRALIZADA
# ============================================================================

@dataclass
class Config:
    """Configuração centralizada do framework."""
    spark_memory: str = "2g"
    pagerank_iterations: int = 20
    criticality_threshold: float = 0.7
    seed: int = 42

config = Config()
print(f" Config carregado: {config}")


# ============================================================================
# CLASSE 2: CARREGADOR DE DADOS
# ============================================================================

class DataGenerator:
    """Gera dados de exemplo para demonstração."""
    
    def __init__(self, spark: SparkSession, num_nodes: int = 50, num_edges: int = 80):
        self.spark = spark
        self.num_nodes = num_nodes
        self.num_edges = num_edges
    
    def create_topology(self) -> 'pyspark.sql.DataFrame':
        """Cria dataset de topologia de rede."""
        np.random.seed(config.seed)
        
        nodes_data = [
            (i, f"node_{i}", 
             np.random.uniform(-90, 90),      # latitude
             np.random.uniform(-180, 180),    # longitude
             np.random.choice(["BR", "US", "EU", "CN", "JP", "SG"]),  # país
             np.random.randint(1000, 65000))  # ASN
            for i in range(self.num_nodes)
        ]
        
        schema = StructType([
            StructField("id", IntegerType(), False),
            StructField("name", StringType(), True),
            StructField("latitude", DoubleType(), True),
            StructField("longitude", DoubleType(), True),
            StructField("country", StringType(), True),
            StructField("asn", IntegerType(), True)
        ])
        
        return self.spark.createDataFrame(nodes_data, schema=schema)
    
    def create_cables(self) -> 'pyspark.sql.DataFrame':
        """Cria dataset de cabos submarinos."""
        np.random.seed(config.seed)
        
        cables_data = [
            (i, f"cable_{i}", 
             np.random.randint(0, self.num_nodes),      # source
             np.random.randint(0, self.num_nodes),      # destination
             np.random.randint(100, 400),               # capacity Gbps
             np.random.randint(1000, 20000))            # length km
            for i in range(self.num_edges)
        ]
        
        schema = StructType([
            StructField("id", IntegerType(), False),
            StructField("name", StringType(), True),
            StructField("src", IntegerType(), True),
            StructField("dst", IntegerType(), True),
            StructField("capacity_gbps", IntegerType(), True),
            StructField("length_km", IntegerType(), True)
        ])
        
        return self.spark.createDataFrame(cables_data, schema=schema)


# ============================================================================
# CLASSE 3: ANALISADOR DE GRAFOS
# ============================================================================

class NetworkAnalyzer:
    """Analisador de propriedades de rede."""
    
    def __init__(self, spark: SparkSession):
        self.spark = spark
    
    def compute_metrics(self, vertices_df, edges_df) -> Dict:
        """Computa métricas básicas de rede."""
        num_vertices = vertices_df.count()
        num_edges = edges_df.count()
        
        degree_df = edges_df.groupBy("src").count()
        avg_degree = degree_df.agg({"count": "avg"}).collect()[0][0] if degree_df.count() > 0 else 0
        
        # Identifica nós críticos (high-degree nodes)
        top_nodes = degree_df.orderBy("count", ascending=False).limit(10)
        
        return {
            'num_vertices': num_vertices,
            'num_edges': num_edges,
            'avg_degree': float(avg_degree) if avg_degree else 0,
            'network_density': (2 * num_edges) / (num_vertices * (num_vertices - 1)) if num_vertices > 1 else 0,
            'top_nodes': top_nodes
        }
    
    def simulate_node_failure(self, vertices_df, edges_df, failed_node_id: int) -> Dict:
        """Simula falha de um nó e calcula impacto."""
        
        # Remove o nó
        vertices_new = vertices_df.filter(col("id") != failed_node_id)
        
        # Remove arestas conectadas ao nó
        edges_new = edges_df.filter(
            (col("src") != failed_node_id) & (col("dst") != failed_node_id)
        )
        
        nodes_affected = vertices_df.count() - vertices_new.count()
        edges_affected = edges_df.count() - edges_new.count()
        connectivity_loss = (nodes_affected / vertices_df.count() * 100) if vertices_df.count() > 0 else 0
        
        return {
            'failed_node': failed_node_id,
            'nodes_affected': nodes_affected + 1,  # +1 for the failed node itself
            'edges_removed': edges_affected,
            'connectivity_loss_percent': connectivity_loss,
            'latency_increase_percent': min(edges_affected / edges_df.count() * 20, 100) if edges_df.count() > 0 else 0
        }


print(" Classes do framework definidas com sucesso!")

##  Passo 4: Inicializar SparkSession

In [ ]:
# Cria SparkSession
spark = SparkSession.builder \
    .appName("RMIFO-Demo") \
    .master("local[*]") \
    .config("spark.driver.memory", config.spark_memory) \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# Reduz verbosidade de logs
spark.sparkContext.setLogLevel("ERROR")

print(f"""
 SparkSession inicializado!
  Versão: {spark.version}
  Master: local[*]
  Memória: {config.spark_memory}
""")

# Inicializa gerador de dados
data_gen = DataGenerator(spark, num_nodes=50, num_edges=80)
print(" Data Generator pronto para gerar dados de exemplo")

##  Passo 5: Carregar Datasets de Exemplo

Agora vamos carregar dados de topologia de rede e cabos submarinos:

In [ ]:
print("⏳ Gerando dados de topologia...")
topology_df = data_gen.create_topology()

print("⏳ Gerando dados de cabos submarinos...")
cables_df = data_gen.create_cables()

print(f"""
 DADOS CARREGADOS COM SUCESSO!

 Estatísticas:
   Nós de topologia: {topology_df.count()}
   Cabos submarinos: {cables_df.count()}
  
{'='*60}
""")

# Exibe amostra de topologia
print(" Amostra de Topologia (primeiros 5 nós):")
topology_df.show(5, truncate=False)

print("\n Amostra de Cabos Submarinos (primeiros 5 cabos):")
cables_df.show(5, truncate=False)

##  Passo 6: Análise de Rede - Métricas & Criticidade

Vamos analisar a topologia de rede e identificar nós críticos:

In [ ]:
# Inicializa analisador
analyzer = NetworkAnalyzer(spark)

# Computa métricas
print(" Computando métricas de rede...")
metrics = analyzer.compute_metrics(topology_df, cables_df)

print(f"""
{'='*60}
MÉTRICAS DA REDE
{'='*60}
 Vértices (Nós): {metrics['num_vertices']}
 Arestas (Cabos): {metrics['num_edges']}
 Grau médio: {metrics['avg_degree']:.2f}
 Densidade de rede: {metrics['network_density']:.4f}
{'='*60}
""")

# Nós críticos (high-degree nodes)
print("  TOP 10 NÓS CRÍTICOS (Por número de conexões):")
top_nodes_df = metrics['top_nodes'].withColumnRenamed("src", "node_id").withColumnRenamed("count", "connections")
top_nodes_df.show(10, truncate=False)

##  Passo 7: Simulação de Falhas - Quantificando Impactos

Vamos simular o que aconteceria se nós críticos falhassem:

In [ ]:
print(f"{'='*60}")
print("SIMULAÇÃO DE FALHAS")
print(f"{'='*60}\n")

# Simula falha de diferentes nós
failure_scenarios = [0, 1, 5, 10, 15]
failure_impacts = []

for node_id in failure_scenarios:
    print(f"Simulando falha do nó {node_id}...")
    impact = analyzer.simulate_node_failure(topology_df, cables_df, node_id)
    failure_impacts.append(impact)
    print(f"   Concluído\n")

# Exibe resultados
print(f"\n{'='*60}")
print("RESULTADOS DAS SIMULAÇÕES")
print(f"{'='*60}\n")

for impact in failure_impacts:
    print(f"""
 Falha do Nó {impact['failed_node']}:
    Nós afetados: {impact['nodes_affected']}
    Cabos removidos: {impact['edges_removed']}
    Perda de conectividade: {impact['connectivity_loss_percent']:.2f}%
    Aumento estimado de latência: {impact['latency_increase_percent']:.2f}%
    """)

print("="*60)

##  Passo 8: Visualização de Resultados

Vamos visualizar os impactos das falhas em gráficos:

In [ ]:
# Prepara dados para visualização
failure_df = pd.DataFrame(failure_impacts)

# Cria figura com subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(' ANÁLISE DE IMPACTO DE FALHAS DE NÓS', fontsize=16, fontweight='bold')

# Gráfico 1: Nós afetados
ax1 = axes[0, 0]
bars1 = ax1.bar(failure_df['failed_node'].astype(str), failure_df['nodes_affected'], color='#FF6B6B')
ax1.set_xlabel('Nó que Falhou', fontweight='bold')
ax1.set_ylabel('Nós Afetados', fontweight='bold')
ax1.set_title(' Nós Afetados por Falha')
ax1.grid(alpha=0.3)

# Gráfico 2: Cabos removidos
ax2 = axes[0, 1]
bars2 = ax2.bar(failure_df['failed_node'].astype(str), failure_df['edges_removed'], color='#4ECDC4')
ax2.set_xlabel('Nó que Falhou', fontweight='bold')
ax2.set_ylabel('Cabos Removidos', fontweight='bold')
ax2.set_title(' Cabos Removidos por Falha')
ax2.grid(alpha=0.3)

# Gráfico 3: Perda de conectividade
ax3 = axes[1, 0]
bars3 = ax3.bar(failure_df['failed_node'].astype(str), failure_df['connectivity_loss_percent'], 
                color='#FFE66D')
ax3.set_xlabel('Nó que Falhou', fontweight='bold')
ax3.set_ylabel('Perda de Conectividade (%)', fontweight='bold')
ax3.set_title(' Perda de Conectividade')
ax3.grid(alpha=0.3)

# Gráfico 4: Aumento de latência
ax4 = axes[1, 1]
bars4 = ax4.bar(failure_df['failed_node'].astype(str), failure_df['latency_increase_percent'], 
                color='#95E1D3')
ax4.set_xlabel('Nó que Falhou', fontweight='bold')
ax4.set_ylabel('Aumento de Latência (%)', fontweight='bold')
ax4.set_title('⏱ Aumento Estimado de Latência')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(" Visualizações geradas com sucesso!")

##  Resumo Executivo - Tabela de Resultados

In [ ]:
# Cria dataframe formatado para exibição
summary_df = pd.DataFrame({
    'Nó Crítico': failure_df['failed_node'],
    'Nós Afetados': failure_df['nodes_affected'],
    'Cabos Removidos': failure_df['edges_removed'],
    'Perda Conectividade (%)': failure_df['connectivity_loss_percent'].round(2),
    'Aumento Latência (%)': failure_df['latency_increase_percent'].round(2)
})

print("\n" + "="*80)
print("TABELA CONSOLIDADA - IMPACTO DE FALHAS")
print("="*80 + "\n")
print(summary_df.to_string(index=False))
print("\n" + "="*80)

# Estatísticas agregadas
print("\n ESTATÍSTICAS AGREGADAS:")
print(f"   Média de nós afetados: {failure_df['nodes_affected'].mean():.2f}")
print(f"   Média de cabos removidos: {failure_df['edges_removed'].mean():.2f}")
print(f"   Perda média de conectividade: {failure_df['connectivity_loss_percent'].mean():.2f}%")
print(f"   Aumento médio de latência: {failure_df['latency_increase_percent'].mean():.2f}%")
print(f"   Pior cenário - Nó: {failure_df.loc[failure_df['nodes_affected'].idxmax(), 'failed_node']:.0f} com {failure_df['nodes_affected'].max()} nós afetados")

##  Passo 9: Análise Comparativa de Cenários

Comparamos o impacto relativo entre diferentes falhas:

In [ ]:
# Normaliza dados para heatmap
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
metrics_normalized = scaler.fit_transform(failure_df[[
    'nodes_affected', 'edges_removed', 'connectivity_loss_percent', 'latency_increase_percent'
]])

# Cria heatmap
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(metrics_normalized.T, cmap='YlOrRd', aspect='auto')

# Labels
ax.set_xticks(range(len(failure_df)))
ax.set_xticklabels([f"Nó {int(n)}" for n in failure_df['failed_node']], fontweight='bold')
ax.set_yticks(range(4))
ax.set_yticklabels(['Nós Afetados', 'Cabos Removidos', 'Perda Conectividade', 'Aumento Latência'], fontweight='bold')

# Adiciona valores nas células
for i in range(len(failure_df)):
    for j in range(4):
        text = ax.text(i, j, f'{metrics_normalized[i, j]:.2f}',
                      ha="center", va="center", color="black", fontweight='bold', fontsize=9)

ax.set_title(' Heatmap de Impacto Normalizado das Falhas', fontweight='bold', fontsize=14, pad=20)

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Severidade (0=Baixa, 1=Alta)', fontweight='bold')

plt.tight_layout()
plt.show()

print(" Heatmap de análise comparativa gerado!")

##  Arquitetura Modularizada do Framework

O framework RMIFO foi projetado com **máxima modularização** e extensibilidade:

```

           RMIFO - Network Resilience Engine              

                                                          
           
     Config            Data             Graph      
   Centralizado  ->   Generator    ->  Processor   
           
                                                         
           
    Failure      <-   Network      <-   Metrics &  
    Simulator         Analyzer         Criticality 
           
                                                          
                                          
     Results &                                          
     Export                                             
                                          
                                                          

```

### Vantagens:
-  **Modular**: Cada componente independente e reutilizável
-  **Escalável**: Funciona com dados pequenos até PB (BigData)
-  **Extensível**: Fácil adicionar novos algoritmos e análises
-  **Configurável**: Arquivo YAML centralizado
-  **Distribuído**: Integrado com Apache Spark para processamento paralelo

##  Exemplo de Extensão: Novo Analisador Customizado

Aqui mostramos como adicionar uma análise customizada ao framework:

In [ ]:
print("="*60)
print("EXTENSÃO DO FRAMEWORK: CustomEdgeAnalyzer")
print("="*60 + "\n")

# Novo analisador customizado que herda de NetworkAnalyzer
class CustomEdgeAnalyzer(NetworkAnalyzer):
    """Analisador customizado focado em arestas críticas."""
    
    def __init__(self, spark):
        super().__init__(spark)
        print(" CustomEdgeAnalyzer inicializado")
    
    def identify_critical_cables(self, edges_df, topology_df, threshold=0.7):
        """Identifica cabos críticos baseado em comprimento e capacidade."""
        # Classifica por comprimento (quanto mais longo, mais crítico)
        from pyspark.sql.window import Window
        from pyspark.sql.functions import row_number, desc
        
        window = Window.orderBy(desc("length_km"))
        critical = edges_df.withColumn("rank", row_number().over(window)) \
                          .filter(col("rank") <= int(edges_df.count() * (1 - threshold)))
        
        return critical
    
    def calculate_redundancy(self, vertices_df, edges_df):
        """Calcula nível de redundância da rede."""
        num_verts = vertices_df.count()
        num_edges = edges_df.count()
        
        # Redundância = grau médio / grau mínimo necessário para conectividade
        avg_degree = (2 * num_edges) / num_verts if num_verts > 0 else 0
        min_degree_for_connectivity = 2  # Mínimo para uma árvore conectada
        
        redundancy_score = avg_degree / min_degree_for_connectivity
        
        return {
            'redundancy_score': redundancy_score,
            'avg_degree': avg_degree,
            'is_redundant': redundancy_score > 2.0
        }


# Usa o novo analisador
custom_analyzer = CustomEdgeAnalyzer(spark)

# Identifica cabos críticos
print(" Identificando cabos críticos...")
critical_cables = custom_analyzer.identify_critical_cables(cables_df, topology_df, threshold=0.7)
print(f" {critical_cables.count()} cabos críticos identificados\n")

print("Cabos críticos (Top 5 mais longos):")
critical_cables.orderBy("length_km", ascending=False).show(5, truncate=False)

# Calcula redundância da rede
print("\n Análise de Redundância:")
redundancy = custom_analyzer.calculate_redundancy(topology_df, cables_df)
print(f"""
   Score de redundância: {redundancy['redundancy_score']:.2f}
   Grau médio: {redundancy['avg_degree']:.2f}
   Rede está redundante: {' SIM' if redundancy['is_redundant'] else ' NÃO'}
""")

##  Conclusões e Recomendações

Com base na análise realizada neste notebook:

In [ ]:
print(f"""
{'='*70}
 RELATÓRIO EXECUTIVO - ANÁLISE DE RESILIÊNCIA DE REDE
{'='*70}

1⃣ TOPOLOGIA DA REDE:
    Total de nós: {topology_df.count()}
    Total de cabos: {cables_df.count()}
    Densidade: {metrics['network_density']:.4f}
    Esta é uma rede {'DENSA' if metrics['network_density'] > 0.5 else 'ESPARSA'}

2⃣ NÓS CRÍTICOS IDENTIFICADOS:
    {len(failure_scenarios)} nós simulados
    Impacto máximo: {failure_df['nodes_affected'].max()} nós afetados
    Impacto mínimo: {failure_df['nodes_affected'].min()} nós afetados
    Impacto médio: {failure_df['nodes_affected'].mean():.1f} nós afetados

3⃣ VULNERABILIDADES IDENTIFICADAS:
    Perda máxima de conectividade: {failure_df['connectivity_loss_percent'].max():.2f}%
    Aumento máximo de latência: {failure_df['latency_increase_percent'].max():.2f}%
    Nó mais crítico: {failure_df.loc[failure_df['nodes_affected'].idxmax(), 'failed_node']:.0f}

4⃣ REDUNDÂNCIA DA REDE:
    Score de redundância: {redundancy['redundancy_score']:.2f}
    Status: {' ADEQUADA' if redundancy['is_redundant'] else '  INSUFICIENTE'}

5⃣ RECOMENDAÇÕES:
   {' Adicionar links redundantes' if not redundancy['is_redundant'] else ' Nível de redundância aceitável'}
    Implementar monitoramento em nós críticos ({failure_df.loc[0, 'failed_node']:.0f}, etc)
    Diversificar rotas de cabos submarinos
    Implementar failover automático

{'='*70}
""")

print(f"\n ANÁLISE COMPLETA EM {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(" Para mais informações, consulte a documentação do projeto.")

##  Próximos Passos

### Para o Orientador/Demonstração:
1. **Execute este notebook no Google Colab** → Pressione "Conectar" e execute célula por célula
2. **Veja o código em ação** → Dados sendo processados em time real
3. **Analise os resultados** → Gráficos e tabelas interativas
4. **Explore extensibilidade** → Veja como adicionar novos analisadores

### Para Desenvolvimento:
-  Adicione dados reais (CAIDA ITDK, TeleGeography)
-  Implemente análise de falhas em cascata
-  Integre visualizações interativas (Plotly, Folium)
-  Desenvolva modelos ML para prever criticidade
-  Crie API REST para queriess em tempo real

### Recursos do Projeto Completo:
-  **Código modularizado**: `src/` com 6 módulos principais
-  **Configuração centralizada**: `config/main_config.yaml`
-  **Pipeline completo**: `main.py` para execução automática
-  **Documentação**: `README_USAGE.md` com exemplos
-  **Notebooks adicionais**: `02_rmifo_quick_start.ipynb`

---

**Versão**: 0.1.0  
**Data**: 2024  
**Framework**: PySpark + GraphFrames  
**Autor**: Network Resilience Team

>  Este notebook foi otimizado para **Google Colab**. Basta copiar o link e compartilhar com o orientador!